In [ ]:
from pathlib import Path
import csv
import gc
import numpy as np
import pyarrow.parquet as pq
from tokenizers import Tokenizer
from tqdm.auto import tqdm
import hashlib

# ============================================================
# CONFIG
# ============================================================

p_val = 1
p_code_corpus = 5
seed = 42

max_reference_tokens = 8192

batch_size = 512  # kleiner = weniger RAM, größer = schneller

tokenizer_name = "/home/jonas/models/huggingface/hub/gemma-4-31B-it-unsloth-bnb-4bit"
tokenizer = Tokenizer.from_file(str(Path(tokenizer_name) / "tokenizer.json"))

datasets = [
    {
        "source": "datikz_v4",
        "path": Path("/home/jonas/Datasets/TikZ/parquet_files/DaTikZ-v4"),
        "description_col": "vlm_description",
        "code_col": "tikz_code",
        "image_col": "png_image",
        "sample_percent": 100,
    },
    {
        "source": "geotikz_bridge_base",
        "path": Path("/home/jonas/Datasets/TikZ/parquet_files/geotikz_bridge_base"),
        "description_col": None,
        "code_col": "response",
        "image_col": "image",
        "sample_percent": 25,
    },
    {
        "source": "our_dataset",
        "path": Path("/home/jonas/Datasets/TikZ/parquet_files/our_dataset_train"),
        "description_col": None,
        "code_col": "code",
        "image_col": "image",
        "sample_percent": 100,
    },
]

output_dir = Path("/home/jonas/Datasets/TikZ/train-big")

image_folder = output_dir / "images"
code_folder = output_dir / "references"
description_folder = output_dir / "vlm_descriptions"
code_corpus_folder = output_dir / "code_corpus"


# ============================================================
# SETUP
# ============================================================

for folder in [image_folder, code_folder, description_folder, code_corpus_folder]:
    folder.mkdir(parents=True, exist_ok=True)

if p_val + p_code_corpus >= 100:
    raise ValueError("p_val + p_code_corpus muss kleiner als 100 sein.")

print(f"Output folder: {output_dir}")
print(f"Batch size: {batch_size}")
print(f"Max reference tokens: {max_reference_tokens}")
print(f"Validation split: {p_val}%")
print(f"Code corpus split: {p_code_corpus}%")


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_text(value) -> str:
    if value is None:
        return ""
    return str(value)


def is_empty_code(text: str) -> bool:
    text = text.strip()
    return not text or text.lower() == "nan"


def token_count(text: str) -> int:
    return len(tokenizer.encode(str(text), add_special_tokens=False).ids)


def write_png(value, path: Path):
    if isinstance(value, dict):
        value = value.get("bytes")

    if isinstance(value, memoryview):
        value = value.tobytes()

    if isinstance(value, bytearray):
        value = bytes(value)

    if isinstance(value, np.bytes_):
        value = bytes(value)

    if not isinstance(value, bytes):
        raise TypeError(f"Unsupported image type: {type(value)}")

    path.write_bytes(value)

def keep_by_sample_percent(source: str, source_idx: int, sample_percent: float, seed: int) -> bool:
    if sample_percent >= 100:
        return True

    if sample_percent <= 0:
        return False

    key = f"{seed}:{source}:{source_idx}".encode("utf-8")
    value = int.from_bytes(hashlib.blake2b(key, digest_size=8).digest(), "big")
    random_value = value / 2**64

    return random_value < (sample_percent / 100)
# ============================================================
# FIRST PASS: COUNT VALID ITEMS ONLY
# ============================================================

valid_count = 0
total_seen = 0
too_long = 0
empty_code = 0
sampled_out = 0

print("\nFirst pass: sampling and filtering by token length...")

for dataset in datasets:
    source = dataset["source"]
    sample_percent = dataset.get("sample_percent", 100)

    if not (0 <= sample_percent <= 100):
        raise ValueError(f"sample_percent für {source} muss zwischen 0 und 100 liegen.")

    source_idx = 0
    parquet_files = sorted(dataset["path"].glob("*.parquet"))

    print(f"\nSource: {source}")
    print(f"Parquet files: {len(parquet_files)}")

    for parquet_file in tqdm(parquet_files, desc=f"Filtering {source}"):
        pf = pq.ParquetFile(parquet_file)
        file_rows = pf.metadata.num_rows

        with tqdm(total=file_rows, desc=parquet_file.name, leave=False) as pbar:
            for batch in pf.iter_batches(
                batch_size=batch_size,
                columns=[dataset["code_col"]],
            ):
                data = batch.to_pydict()
                codes = data[dataset["code_col"]]

                for code_value in codes:
                    row_idx = source_idx
                    source_idx += 1

                    code_text = normalize_text(code_value)
                    total_seen += 1

                    if not keep_by_sample_percent(source, row_idx, sample_percent, seed):
                        sampled_out += 1
                        continue

                    if is_empty_code(code_text):
                        empty_code += 1
                    elif token_count(code_text) <= max_reference_tokens:
                        valid_count += 1
                    else:
                        too_long += 1

                pbar.update(len(codes))

        gc.collect()

print("\nFiltering summary:")
print(f"Total rows seen: {total_seen}")
print(f"Sampled out: {sampled_out}")
print(f"Rows after sampling: {total_seen - sampled_out}")
print(f"Valid rows: {valid_count}")
print(f"Filtered too long: {too_long}")
print(f"Filtered empty code: {empty_code}")


# ============================================================
# SPLIT VALID ITEMS INTO TRAIN / VAL / CODE CORPUS
# ============================================================

print("\nSplitting valid items...")

rng = np.random.default_rng(seed)

n_code_corpus = round(valid_count * p_code_corpus / 100)
n_val = round(valid_count * p_val / 100)
n_train = valid_count - n_code_corpus - n_val

# 0 = train, 1 = val, 2 = code_corpus
split_labels = np.zeros(valid_count, dtype=np.uint8)

indices = rng.permutation(valid_count)

split_labels[indices[:n_code_corpus]] = 2
split_labels[indices[n_code_corpus:n_code_corpus + n_val]] = 1

del indices
gc.collect()

print(f"Target train rows: {n_train}")
print(f"Target val rows: {n_val}")
print(f"Target code corpus rows: {n_code_corpus}")


# ============================================================
# SECOND PASS: WRITE FILES AND MANIFESTS BATCHWISE
# ============================================================

manifest_columns = [
    "code_path",
    "image_path",
    "vlm_description_path",
    "reference_image",
    "reference_code",
    "input_image_format_for_vlm",
    "vlm_description",
]

train_manifest_path = output_dir / "manifest_train.csv"
val_manifest_path = output_dir / "manifest_val.csv"

written_train = 0
written_val = 0
written_code_corpus = 0
valid_idx = 0

print("\nSecond pass: writing files and manifests...")

with train_manifest_path.open("w", newline="", encoding="utf-8") as train_file, \
     val_manifest_path.open("w", newline="", encoding="utf-8") as val_file:

    train_writer = csv.DictWriter(train_file, fieldnames=manifest_columns)
    val_writer = csv.DictWriter(val_file, fieldnames=manifest_columns)

    train_writer.writeheader()
    val_writer.writeheader()

    for dataset in datasets:
        source = dataset["source"]
        sample_percent = dataset.get("sample_percent", 100)
        source_idx = 0
        parquet_files = sorted(dataset["path"].glob("*.parquet"))

        print(f"\nSource: {source}")

        for parquet_file in tqdm(parquet_files, desc=f"Writing {source}"):
            pf = pq.ParquetFile(parquet_file)
            file_rows = pf.metadata.num_rows

            columns = [
                dataset["code_col"],
                dataset["image_col"],
            ]

            if dataset["description_col"] is not None:
                columns.append(dataset["description_col"])

            with tqdm(total=file_rows, desc=parquet_file.name, leave=False) as pbar:
                for batch in pf.iter_batches(batch_size=batch_size, columns=columns):
                    data = batch.to_pydict()

                    codes = data[dataset["code_col"]]
                    images = data[dataset["image_col"]]

                    if dataset["description_col"] is None:
                        descriptions = [""] * len(codes)
                    else:
                        descriptions = data[dataset["description_col"]]

                    for description_value, code_value, image_value in zip(descriptions, codes, images):
                        row_idx = source_idx
                        stem = f"{source}_{source_idx:08d}"
                        source_idx += 1

                        if not keep_by_sample_percent(source, row_idx, sample_percent, seed):
                            continue

                        code_text = normalize_text(code_value)

                        if is_empty_code(code_text):
                            continue

                        if token_count(code_text) > max_reference_tokens:
                            continue

                        split = split_labels[valid_idx]
                        valid_idx += 1

                        if split == 2:
                            code_corpus_path = code_corpus_folder / f"{stem}.txt"
                            code_corpus_path.write_text(code_text, encoding="utf-8")
                            written_code_corpus += 1
                            continue

                        generated_image = image_folder / f"{stem}.png"
                        code_path = code_folder / f"{stem}.txt"
                        description_path = description_folder / f"{stem}.txt"

                        write_png(image_value, generated_image)
                        code_path.write_text(code_text, encoding="utf-8")
                        description_path.write_text(
                            normalize_text(description_value),
                            encoding="utf-8",
                        )

                        manifest_row = {
                            "code_path": f"/data/references/{code_path.name}",
                            "image_path": f"/data/images/{generated_image.name}",
                            "vlm_description_path": f"/data/vlm_descriptions/{description_path.name}",
                            "reference_image": f"/images/{generated_image.name}",
                            "reference_code": f"/references/{code_path.name}",
                            "input_image_format_for_vlm": f"file:///images/{generated_image.name}",
                            "vlm_description": f"file:///vlm_descriptions/{description_path.name}",
                        }

                        if split == 1:
                            val_writer.writerow(manifest_row)
                            written_val += 1
                        else:
                            train_writer.writerow(manifest_row)
                            written_train += 1

                    pbar.update(batch.num_rows)

            gc.collect()


# ============================================================
# CHECKS
# ============================================================

if valid_idx != valid_count:
    raise RuntimeError(
        f"Valid count mismatch: first pass found {valid_count}, "
        f"second pass wrote/processed {valid_idx}"
    )


# ============================================================
# SUMMARY
# ============================================================

print("\nDone.")
print(f"Total rows seen: {total_seen}")
print(f"Valid rows: {valid_count}")
print(f"Filtered too long: {too_long}")
print(f"Filtered empty code: {empty_code}")
print(f"Sampled out: {sampled_out}")
print(f"\nSource: {source}")
print(f"Sample percent: {sample_percent}%")
print(f"Parquet files: {len(parquet_files)}")

print()
print(f"Train rows: {written_train}")
print(f"Val rows: {written_val}")
print(f"Code corpus rows: {written_code_corpus}")

print()
print(f"Train manifest: {train_manifest_path}")
print(f"Val manifest: {val_manifest_path}")
print(f"Code corpus folder: {code_corpus_folder}")
print(f"Output folder: {output_dir}")

Output folder: /home/jonas/Datasets/TikZ/train-big
Batch size: 512
Max reference tokens: 8192
Validation split: 1%
Code corpus split: 5%

First pass: sampling and filtering by token length...

Source: datikz_v4
Parquet files: 86


Filtering datikz_v4: 100%|██████████| 86/86 [01:57<00:00,  1.37s/it]



Source: geotikz_bridge_base
Parquet files: 4


Filtering geotikz_bridge_base: 100%|██████████| 4/4 [02:00<00:00, 30.16s/it]



Source: our_dataset
Parquet files: 1


...


Source: geotikz_bridge_base


Writing geotikz_bridge_base: 100%|██████████| 4/4 [03:00<00:00, 45.07s/it]



Source: our_dataset


Writing our_dataset: 100%|██████████| 1/1 [00:37<00:00, 37.15s/it]


Done.
Total rows seen: 2975286
Valid rows: 1005618
Filtered too long: 7191
Filtered empty code: 0
Sampled out: 1962477

Source: our_dataset
Sample percent: 100%
Parquet files: 1

Train rows: 945281
Val rows: 10056
Code corpus rows: 50281

Train manifest: /home/jonas/Datasets/TikZ/train-big/manifest_train.csv
Val manifest: /home/jonas/Datasets/TikZ/train-big/manifest_val.csv
Code corpus folder: /home/jonas/Datasets/TikZ/train-big/code_corpus
Output folder: /home/jonas/Datasets/TikZ/train-big
